In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from transformers import pipeline
from sklearn.metrics import (
    log_loss, accuracy_score,
    precision_score, recall_score, confusion_matrix
)

In [34]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
BASE_DIR   = "/Users/kaatsandoval/code/katiarojas87/final-project/raw_data/suumo_images"
THRESHOLD  = 0.65

os.chdir("/Users/kaatsandoval/code/katiarojas87/final-project")

%store -r df_cleaned

In [35]:
# ── 1. LOAD MODEL ─────────────────────────────────────────────────────────────
clip = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=0 if torch.cuda.is_available() else -1,
)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
# ── 2. LABEL TEMPLATES ────────────────────────────────────────────────────────
# Each attribute maps to an ORDERED list of labels.
# Index 0  →  positive class (score returned = P(label[0]))
# You can add as many candidate labels as you like; the model will softmax
# across all of them.  Just keep label[0] as the "positive" one.
LABEL_TEMPLATES: dict[str, list[str]] = {
    "luxury": [
            f"a photo of an expensive, luxurious {room} with premium finishes",
            f"a photo of a cheap, basic {room} with low-quality materials",
    ],
    "brightness": [
        "a photo of a {room} with large windows and bright natural sunlight filling the space",  # positive
        "a photo of a {room} lit only by artificial ceiling lights, no sunlight",                # negative-1
        "a photo of a dark {room} with curtains closed and poor lighting",                       # negative-2
    ],
    "condition": [
        "a photo of a brand new {room} with fresh white walls and modern clean fixtures",        # positive — tighter
        "a photo of a {room} with slightly worn surfaces and older fixtures",                    # negative-1
        "a photo of a {room} with cracked walls, stains, and visibly damaged surfaces",         # negative-2
    ],
}

NameError: name 'room' is not defined

In [37]:
def build_labels(room_type, attribute: str) -> list[str]:
    if attribute not in LABEL_TEMPLATES:
        raise ValueError(f"Unknown attribute '{attribute}'. Choose from: {list(LABEL_TEMPLATES)}")
    room_str = str(room_type) if pd.notna(room_type) else "room"  # ← only change
    return [t.replace("{room}", room_str) for t in LABEL_TEMPLATES[attribute]]

In [38]:
# ── 3. SCORING ────────────────────────────────────────────────────────────────
def get_score(image_path: str, labels: list[str], clip_pipeline) -> float:
    """
    Returns the probability assigned to labels[0] (the positive class).
    Works with any number of candidate labels ≥ 2.
    """
    image   = Image.open(image_path).convert("RGB")
    results = clip_pipeline(image, candidate_labels=labels)
    scores  = {r["label"]: r["score"] for r in results}
    return scores[labels[0]]


In [39]:
# ── 4. APPLY SCORES ───────────────────────────────────────────────────────────
ATTRIBUTES: dict[str, str] = {
    "luxury":     "score_lux",
    "brightness": "score_bright",
    "condition":  "score_cond",
}

for attribute, score_col in ATTRIBUTES.items():
    df_cleaned[score_col] = df_cleaned.apply(
        lambda row, attr=attribute: get_score(
            row["image_path"],
            build_labels(row["RoomType"], attr),
            clip,
        ),
        axis=1,
    )

# Quick sanity check
print(df_cleaned[[
    "image_path", "RoomType",
    "Luxury", "Brightness", "Condition",
    "score_lux", "score_bright", "score_cond",
]].head())

                                          image_path     RoomType Luxury  \
0  raw_data/suumo_images/20277732/20277732_eaa711...  living room    yes   
2  raw_data/suumo_images/20277732/20277732_274be4...  living room    yes   
3  raw_data/suumo_images/20277732/20277732_c35b74...      kitchen    yes   
4  raw_data/suumo_images/20277732/20277732_bbd7ff...      bedroom     no   
5  raw_data/suumo_images/20277732/20277732_8b8db6...     bathroom    yes   

  Brightness Condition  score_lux  score_bright  score_cond  
0        yes       new   0.372089      0.993086    0.673374  
2        yes       new   0.156122      0.855128    0.306381  
3        yes       new   0.371793      0.982175    0.338951  
4        yes       old   0.130328      0.962551    0.716400  
5        yes       new   0.697483      0.993527    0.869622  


In [40]:
# ── 5. EVALUATION ─────────────────────────────────────────────────────────────
def get_accuracy(
    df: pd.DataFrame,
    threshold: float = THRESHOLD,
) -> dict[str, dict]:
    """
    Evaluate CLIP scores against ground-truth labels.
    Uses a single shared threshold (default 0.55) for all attributes.
    """
    configs = [
        # (display name,  score column,   ground-truth boolean series)
        ("Luxury",     "score_lux",    df["Luxury"]          == "yes"),
        ("Brightness", "score_bright", df["Brightness"]      == "yes"),
        ("Condition",  "score_cond",   df["Condition"] == "new"),
    ]

    results: dict[str, dict] = {}

    for label, score_col, y_true_bool in configs:
        y_true = y_true_bool.astype(int)
        y_prob = df[score_col]
        y_pred = (y_prob > threshold).astype(int)

        results[label] = {
            "log_loss":         round(log_loss(y_true, y_prob), 4),
            "accuracy":         round(accuracy_score(y_true, y_pred), 4),
            "precision":        round(precision_score(y_true, y_pred, zero_division=0), 4),
            "recall":           round(recall_score(y_true, y_pred, zero_division=0), 4),
            "confusion_matrix": confusion_matrix(y_true, y_pred).ravel().tolist(),
            # confusion matrix layout: [TN, FP, FN, TP]
        }

    return results


In [33]:
# ── 6. PRINT RESULTS ──────────────────────────────────────────────────────────
metrics = get_accuracy(df_cleaned)

for attr, m in metrics.items():
    print(f"\n{'─'*40}")
    print(f"  {attr.upper()}")
    print(f"{'─'*40}")
    tn, fp, fn, tp = m["confusion_matrix"]
    for k, v in m.items():
        if k != "confusion_matrix":
            print(f"  {k:<16}: {v}")
    print(f"  {'TN/FP/FN/TP':<16}: {tn} / {fp} / {fn} / {tp}")


────────────────────────────────────────
  LUXURY
────────────────────────────────────────
  log_loss        : 0.9305
  accuracy        : 0.5254
  precision       : 0.6667
  recall          : 0.069
  TN/FP/FN/TP     : 29 / 1 / 27 / 2

────────────────────────────────────────
  BRIGHTNESS
────────────────────────────────────────
  log_loss        : 1.1339
  accuracy        : 0.5254
  precision       : 0.5088
  recall          : 1.0
  TN/FP/FN/TP     : 2 / 28 / 0 / 29

────────────────────────────────────────
  CONDITION
────────────────────────────────────────
  log_loss        : 0.7794
  accuracy        : 0.5424
  precision       : 0.525
  recall          : 0.7241
  TN/FP/FN/TP     : 11 / 19 / 8 / 21
